# Seller Performance — Exploratory Data Analysis
**Domain:** Seller Performance | **Developer:** Huey Ling

This notebook analyses Olist seller behaviour using the Gold tables built by the batch pipeline.

## Contents
1. Setup & connection
2. Dataset overview
3. Revenue analysis
4. Top Sellers Overview
5. At-risk Sellers Overview
6. Delivery Performance
7. Marketplace concentration (Gini / Lorenz)
8. Power seller & at-risk identification

## 1. Setup & connection

In [1]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from shared.utils import dev_config_path, make_bq_client_getter, run_query, qualified_table

_get_client = make_bq_client_getter(dev_config_path("huey_ling"))
client, cfg, err = _get_client()
if err:
    raise RuntimeError(f"GCP connection failed: {err}")
print(f"Connected to project: {cfg['project_id']}  dataset: {cfg['dataset']}")

Connected to project: my-shl-project-olist  dataset: olist_gold


## 2. Dataset overview

In [3]:
# Load full Dim_Sellers and a sample of Fact_Orders
sellers = run_query(client, f"SELECT * FROM {qualified_table(cfg, 'Dim_Sellers')}")
print(f"Dim_Sellers: {len(sellers):,} rows")
sellers.head(3)

Dim_Sellers: 3,095 rows


,seller_id,seller_city,seller_state,latitude,longitude,total_orders,total_revenue,avg_order_value,avg_review_score,on_time_rate,unique_products,power_seller,at_risk
0,1ca7077d890b907f89be8c954a02686a,santana de parnaiba,SP,-23.418614,-46.958876,115,13341.57,116.01,2.33,0.778,21,False,True
1,2eb70248d66e0e3ef83659f71b244378,campinas,SP,-22.914020,-47.013535,202,42628.61,211.03,2.70,0.861,33,False,True
2,54965bbe3e4f07ae045b90b0b8541f52,foz do iguacu,PR,-25.541845,-54.584820,78,10961.30,140.53,3.00,0.699,53,False,True


In [4]:
orders = run_query(client, f"""
    SELECT seller_id, order_id, order_status, order_purchase_timestamp,
           items_total, payment_value, review_score, on_time_delivery, delivery_delay_days
    FROM {qualified_table(cfg, 'Fact_Orders')}
""")
print(f"Fact_Orders: {len(orders):,} rows")
orders.dtypes

Fact_Orders: 10,000 rows


seller_id                                object
order_id                                 object
order_status                             object
order_purchase_timestamp    datetime64[us, UTC]
items_total                             float64
payment_value                           float64
review_score                              Int64
on_time_delivery                        boolean
delivery_delay_days                       Int64
dtype: object

In [5]:
# Active vs inactive sellers
active = sellers[sellers.total_orders > 0]
inactive = sellers[sellers.total_orders == 0]
print(f"Total sellers:    {len(sellers):,}")
print(f"Active sellers:   {len(active):,} ({len(active)/len(sellers)*100:.1f}%)")
print(f"Inactive sellers: {len(inactive):,} ({len(inactive)/len(sellers)*100:.1f}%)")
print(f"Total revenue:    R$ {active.total_revenue.sum():,.0f}")
print(f"Total orders:     {active.total_orders.sum():,.0f}")

Total sellers:    3,095
Active sellers:   3,095 (100.0%)
Inactive sellers: 0 (0.0%)
Total revenue:    R$ 13,591,644
Total orders:     100,010


## 3. Revenue analysis

In [6]:
# Revenue distribution — log-transform first so bins are evenly spaced on log scale
import numpy as np

plot_df = active.copy()
plot_df["log10_revenue"] = np.log10(plot_df["total_revenue"])

fig = px.histogram(
    plot_df, x="log10_revenue", nbins=40,
    title="Revenue Distribution per Seller (log₁₀ scale)",
    labels={"log10_revenue": "Total Revenue (BRL)"},
    color_discrete_sequence=["#FF8C00"],
)
# Replace numeric log10 ticks with BRL labels
tick_vals = [1, 2, 3, 4, 5]   # 10, 100, 1K, 10K, 100K
tick_text = ["R$10", "R$100", "R$1K", "R$10K", "R$100K"]
fig.update_xaxes(tickvals=tick_vals, ticktext=tick_text)
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0D0D0D",
    plot_bgcolor="#1A1A1A",
    yaxis_title="Number of Sellers",
)
fig.show()

In [7]:
# Revenue percentiles
pcts = [50, 75, 90, 95, 99]
print("Revenue percentiles (active sellers):")
for p in pcts:
    print(f"  P{p:2d}: R$ {np.percentile(active.total_revenue, p):,.0f}")
print(f"  Max: R$ {active.total_revenue.max():,.0f}")

Revenue percentiles (active sellers):
  P50: R$ 821
  P75: R$ 3,281
  P90: R$ 9,525
  P95: R$ 16,261
  P99: R$ 55,109
  Max: R$ 229,473


In [8]:
# Top 10 sellers by revenue
top10 = active.nlargest(10, "total_revenue")[["seller_id","seller_state","total_revenue","total_orders","avg_review_score"]]
top10["seller_id"] = top10["seller_id"].str[:8]
top10

,seller_id,seller_state,total_revenue,total_orders,avg_review_score
284,4869f7a5,SP,229472.63,1132,4.13
282,53243585,BA,222776.05,358,4.13
81,4a3ca931,SP,200472.92,1806,3.83
467,fa1c13f2,SP,194042.03,585,4.34
23,7c67e144,SP,187923.89,982,3.49
359,7e93a43e,SP,176431.87,336,4.21
335,da8622b1,SP,160236.57,1314,4.18
395,7a67c85e,SP,141745.53,1160,4.25
180,1025f0e2,SP,138968.55,915,4.00
310,955fee92,SP,135171.70,1287,4.16


### 4. Sellers Overview

In [9]:
# Scatter: total orders (x) vs total revenue (y), bubble size = avg rating
# Symbol distinguishes Power Sellers (star) vs Regular (circle) from Dim_Sellers.power_seller
scatter_df = active.dropna(subset=["avg_review_score"]).copy()
scatter_df["seller_short"] = scatter_df["seller_id"].str[:8]
scatter_df["seller_type"] = scatter_df["power_seller"].map({True: "Power Seller", False: "Regular"})

fig = px.scatter(
    scatter_df,
    x="total_orders",
    y="total_revenue",
    size="avg_review_score",
    size_max=18,
    color="avg_review_score",
    color_continuous_scale=["#FF4444", "#FF8C00", "#00C851"],
    symbol="seller_type",
    symbol_map={"Power Seller": "star", "Regular": "circle"},
    hover_data={
        "seller_short": True,
        "seller_state": True,
        "avg_review_score": ":.2f",
        "seller_type": True,
        "seller_id": False,
    },
    title="Sellers — Orders vs Revenue (bubble size & colour = avg rating, ★ = Power Seller)",
    labels={
        "total_orders": "Total Orders",
        "total_revenue": "Total Revenue (BRL)",
        "avg_review_score": "Avg Rating",
        "seller_type": "Seller Type",
    },
    opacity=0.75,
)
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0D0D0D",
    plot_bgcolor="#1A1A1A",
    coloraxis_colorbar=dict(title="Avg Rating"),
    legend=dict(title="Seller Type", orientation="v", x=1.12, y=0.5),
    xaxis=dict(title=dict(text="Total Orders", standoff=30)),
    annotations=[dict(
        text="★ Power Seller: top 1% revenue AND average review ≥ 4",
        xref="paper", yref="paper",
        x=0, y=-0.3,
        showarrow=False,
        font=dict(size=11, color="#888888"),
        align="center",
    )],
    margin=dict(b=100),
)
fig.show()
print(f"Power sellers shown: {scatter_df['seller_type'].eq('Power Seller').sum():,}")

Power sellers shown: 23


In [11]:
# Dual-axis bar: power sellers — revenue (left y) + total orders (right y)
power_plot = (
    active[active["power_seller"] == True]
    .assign(seller_short=lambda df: df["seller_id"].str[:8])
    .sort_values("total_revenue", ascending=False)
)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=power_plot["seller_short"],
        y=power_plot["total_revenue"],
        name="Total Revenue (BRL)",
        marker_color="#FF8C00",
        opacity=0.85,
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=power_plot["seller_short"],
        y=power_plot["total_orders"],
        name="Total Orders",
        mode="markers+lines",
        marker=dict(color="#FFD700", size=9, symbol="diamond"),
        line=dict(color="#FFD700", width=2, dash="dot"),
    ),
    secondary_y=True,
)

fig.update_layout(
    title=f"Power Sellers ({len(power_plot)}) — Revenue vs Order Volume",
    template="plotly_dark",
    paper_bgcolor="#0D0D0D",
    plot_bgcolor="#1A1A1A",
    legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"),
    xaxis=dict(title=dict(text="Seller ID (first 8 chars)", standoff=30)),
    bargap=0.25,
    annotations=[dict(
        text="★ Power Seller: top 1% revenue AND average review ≥ 4",
        xref="paper", yref="paper",
        x=0.5, y=-0.22,
        showarrow=False,
        font=dict(size=11, color="#888888"),
        align="center",
    )],
    margin=dict(b=100),
)
fig.update_yaxes(title_text="Total Revenue (BRL)", secondary_y=False, tickprefix="R$")
fig.update_yaxes(title_text="Total Orders", secondary_y=True)
fig.show()

In [ ]:
# Dual-axis bar: power sellers — AOV ranking (left y, bars) + total orders (right y, line)
# Sorted by AOV descending to show premium vs mass-market split
aov_orders_plot = (
    active[active["power_seller"] == True]
    .assign(seller_short=lambda df: df["seller_id"].str[:8])
    .sort_values("avg_order_value", ascending=False)
)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=aov_orders_plot["seller_short"],
        y=aov_orders_plot["avg_order_value"],
        name="Avg Order Value (BRL)",
        marker_color="#FF8C00",
        opacity=0.85,
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=aov_orders_plot["seller_short"],
        y=aov_orders_plot["total_orders"],
        name="Total Orders",
        mode="markers+lines",
        marker=dict(color="#FFD700", size=9, symbol="diamond"),
        line=dict(color="#FFD700", width=2, dash="dot"),
    ),
    secondary_y=True,
)

fig.update_layout(
    title=f"Power Sellers ({len(aov_orders_plot)}) — Average Order Value Ranking vs Total Orders",
    template="plotly_dark",
    paper_bgcolor="#0D0D0D",
    plot_bgcolor="#1A1A1A",
    legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"),
    xaxis_title="Seller ID (first 8 chars, ranked by AOV ↓)",
    bargap=0.25,
)
fig.update_yaxes(title_text="Avg Order Value (BRL)", secondary_y=False, tickprefix="R$")
fig.update_yaxes(title_text="Total Orders", secondary_y=True)
fig.show()

### 5. At-risk Sellers

In [12]:
# Scatter: top 20 at-risk sellers by revenue (at_risk == True)
# x = avg_review_score, y = on_time_rate, size = total_revenue (business impact)
risk_df = (
    active[active["at_risk"] == True]
    .dropna(subset=["avg_review_score", "on_time_rate"])
    .nlargest(20, "total_revenue")
    .copy()
)
risk_df["seller_short"] = risk_df["seller_id"].str[:8]
risk_df["breach"] = "None"
risk_df.loc[risk_df["avg_review_score"] < 3.0, "breach"] = "Low Rating"
risk_df.loc[risk_df["on_time_rate"] < 0.70, "breach"] = "High Late Rate"
risk_df.loc[
    (risk_df["avg_review_score"] < 3.0) & (risk_df["on_time_rate"] < 0.70), "breach"
] = "Both"

fig = px.scatter(
    risk_df,
    x="avg_review_score",
    y="on_time_rate",
    color="breach",
    color_discrete_map={
        "Low Rating": "#FFD700",
        "High Late Rate": "#FF8C00",
        "Both": "#FF4444",
    },
    size="total_revenue",
    size_max=20,
    symbol="breach",
    symbol_map={
        "Low Rating": "x",
        "High Late Rate": "triangle-down",
        "Both": "x-open-dot",
    },
    text="seller_short",
    hover_data={
        "seller_short": True,
        "seller_state": True,
        "avg_review_score": ":.2f",
        "on_time_rate": ":.1%",
        "total_revenue": ":,.0f",
        "breach": True,
        "seller_id": False,
    },
    opacity=0.85,
    title="Top 20 At-risk Sellers by Revenue — Review Score vs On-Time Rate (size = revenue)",
    labels={
        "avg_review_score": "Avg Review Score",
        "on_time_rate": "On-Time Delivery Rate",
        "breach": "At-Risk Reason",
    },
)

fig.update_traces(textposition="top center", textfont=dict(size=9, color="#CCCCCC"))

fig.add_vline(x=3.0, line_dash="dash", line_color="#888888",
              annotation_text="Review < 3.0", annotation_position="top right",
              annotation_font_color="#888888")
fig.add_hline(y=0.70, line_dash="dash", line_color="#888888",
              annotation_text="On-time < 70%", annotation_position="bottom right",
              annotation_font_color="#888888")

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0D0D0D",
    plot_bgcolor="#1A1A1A",
    legend=dict(orientation="v", x=1.02, y=0.5),
    xaxis=dict(title=dict(text="Avg Review Score", standoff=30)),
    margin=dict(b=100),
)

# Add footnote after layout is set to avoid annotation duplication from add_vline/add_hline
fig.add_annotation(
    text="At-Risk: Low Rating = average review < 3.0, High Late Rate = on-time rate < 70%",
    xref="paper", yref="paper",
    x=0, y=-0.3,
    showarrow=False,
    font=dict(size=11, color="#888888"),
    align="center",
)

fig.show()

## 6. Delivery Performance

In [13]:
# On-time delivery overview
delivered = orders[orders.on_time_delivery.notna()]
on_time_pct = delivered.on_time_delivery.mean() * 100
print(f"Overall on-time delivery rate: {on_time_pct:.1f}%")

# Delay distribution
delay_df = orders[orders.delivery_delay_days.notna()].copy()
delay_df["delay_clamped"] = delay_df["delivery_delay_days"].clip(-30, 60)
fig = px.histogram(
    delay_df, x="delay_clamped", nbins=60,
    title="Delivery Delay Distribution (days late; negative = early)",
    labels={"delay_clamped": "Delay (days)"},
    color_discrete_sequence=["#FF8C00"],
)
fig.add_vline(x=0, line_dash="dash", line_color="#FFD700", annotation_text="On time")
fig.update_layout(template="plotly_dark", paper_bgcolor="#0D0D0D", plot_bgcolor="#1A1A1A")
fig.show()

Overall on-time delivery rate: 92.9%


In [27]:
# On-time rate by seller state — bar text shows on-time rate % + delayed order count
state_delivery = (
    active.dropna(subset=["on_time_rate"])
    .assign(delayed_orders=lambda df: (df["total_orders"] * (1 - df["on_time_rate"])).round(0).astype(int))
    .groupby("seller_state")
    .agg(
        sellers=("seller_id", "count"),
        avg_on_time=("on_time_rate", "mean"),
        delayed_orders=("delayed_orders", "sum"),
    )
    .reset_index()
    .sort_values("avg_on_time", ascending=False)
)

bar_text = [
    f"{row.avg_on_time:.0%}\n{row.delayed_orders:,}"
    for row in state_delivery.itertuples()
]

fig = px.bar(
    state_delivery, x="seller_state", y="avg_on_time",
    color="avg_on_time",
    color_continuous_scale=["#FF4444", "#FF8C00", "#00C851"],
    text=bar_text,
    custom_data=["sellers", "delayed_orders"],
    title="Average On-Time Delivery Rate by Seller State",
    labels={"avg_on_time": "On-Time Rate", "seller_state": "State"},
)
fig.update_traces(
    textposition="inside",
    textangle=-90,
    textfont=dict(size=10, color="#FFFFFF"),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "On-Time Rate: %{y:.1%}<br>"
        "Delayed Orders: %{customdata[1]:,}<br>"
        "Sellers: %{customdata[0]}<extra></extra>"
    ),
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(template="plotly_dark", paper_bgcolor="#0D0D0D", plot_bgcolor="#1A1A1A")
fig.show()

In [15]:
# Sellers per state
state_stats = (
    active.groupby("seller_state")
    .agg(
        sellers=("seller_id", "count"),
        revenue=("total_revenue", "sum"),
        avg_rating=("avg_review_score", "mean"),
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)
print("Top 10 states by revenue:")
print(state_stats.head(10).to_string(index=False))

Top 10 states by revenue:
seller_state  sellers    revenue  avg_rating
          SP     1849 8753396.21    3.973310
          PR      349 1261887.21    4.044006
          MG      244 1011564.74    4.055656
          RJ      171  843984.22    3.976842
          SC      190  632426.07    4.077053
          RS      129  378559.54    4.057984
          BA       19  285561.56    3.937368
          DF       30   97749.48    4.032333
          PE        9   91493.85    4.123333
          GO       40   66399.21    4.113750


## 7. Marketplace concentration (Lorenz curve)

In [16]:
# Lorenz curve
sorted_rev = np.sort(active["total_revenue"].values)
n = len(sorted_rev)
cum_rev = sorted_rev.cumsum() / sorted_rev.sum()
cum_sellers = np.arange(1, n + 1) / n

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cum_sellers * 100, y=cum_rev * 100,
    name="Actual distribution",
    line=dict(color="#FF8C00", width=2),
    fill="tozeroy", fillcolor="rgba(255,140,0,0.08)",
))
fig.add_trace(go.Scatter(
    x=[0, 100], y=[0, 100],
    name="Perfect equality",
    line=dict(color="#606060", width=1, dash="dash"),
))
# Annotate top 10%
top10_idx = int(n * 0.90)
top10_rev_pct = sorted_rev[top10_idx:].sum() / sorted_rev.sum() * 100
fig.add_annotation(
    x=90, y=top10_rev_pct,
    text=f"Top 10% sellers\n= {top10_rev_pct:.0f}% revenue",
    showarrow=True, arrowhead=2,
    font=dict(color="#FFD700"),
    arrowcolor="#FFD700",
)
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0D0D0D", plot_bgcolor="#1A1A1A",
    title="Lorenz Curve — Marketplace Revenue Concentration",
    xaxis_title="Cumulative % of Sellers",
    yaxis_title="Cumulative % of Revenue",
)
fig.show()

# Gini coefficient
n_f = float(n)
gini = (2 * np.sum((np.arange(1, n + 1) * sorted_rev))) / (n_f * sorted_rev.sum()) - (n_f + 1) / n_f
print(f"Gini coefficient: {gini:.3f}  (0 = perfect equality, 1 = maximum concentration)")
print(f"Top 10% of sellers account for {top10_rev_pct:.1f}% of total revenue")

Gini coefficient: 0.792  (0 = perfect equality, 1 = maximum concentration)
Top 10% of sellers account for 67.6% of total revenue


## 8. Power seller & at-risk identification

In [34]:
# Power sellers
power = active[active["power_seller"] == True]
at_risk = active[active["at_risk"] == True]
print(f"Power sellers (top 1% revenue + rating >= 4): {len(power):,}")
print(f"At-risk sellers (rating < 3 OR on-time < 70%): {len(at_risk):,}")
print(f"Power seller revenue share: {power.total_revenue.sum() / active.total_revenue.sum() * 100:.1f}%")

power[["seller_id", "seller_state", "total_revenue", "avg_order_value", "avg_review_score", "on_time_rate"]]\
    .assign(seller_id=lambda df: df.seller_id.str[:8])\
    .sort_values("total_revenue", ascending=False)\
    .head(10)

Power sellers (top 1% revenue + rating >= 4): 23
At-risk sellers (rating < 3 OR on-time < 70%): 443
Power seller revenue share: 19.7%


,seller_id,seller_state,total_revenue,avg_order_value,avg_review_score,on_time_rate
284,4869f7a5,SP,229472.63,202.71,4.13,0.884
282,53243585,BA,222776.05,622.28,4.13,0.957
467,fa1c13f2,SP,194042.03,331.70,4.34,0.898
359,7e93a43e,SP,176431.87,525.09,4.21,0.944
335,da8622b1,SP,160236.57,121.95,4.18,0.924
395,7a67c85e,SP,141745.53,122.19,4.25,0.941
180,1025f0e2,SP,138968.55,151.88,4.00,0.899
310,955fee92,SP,135171.70,105.03,4.16,0.922
344,46dc3b2c,RJ,128111.19,245.89,4.19,0.935
400,620c87c1,RJ,114774.50,155.10,4.25,0.901


In [37]:
# At-risk breakdown
low_rating = active[active.avg_review_score < 3.0]
high_late = active[(active.on_time_rate < 0.70) & active.on_time_rate.notna()]
both = active[(active.avg_review_score < 3.0) & (active.on_time_rate < 0.70)]
print(f"At-risk breakdown:")
print(f"  Low rating only (< 3.0):      {len(low_rating) - len(both):,}")
print(f"  High late-delivery only (>30%): {len(high_late) - len(both):,}")
print(f"  Both:                          {len(both):,}")

# At-risk by state
at_risk_state = (
    at_risk.groupby("seller_state")
    .size()
    .reset_index(name="at_risk_count")
    .merge(state_stats[["seller_state", "sellers"]], on="seller_state")
    .assign(at_risk_pct=lambda df: df.at_risk_count / df.sellers * 100)
    .sort_values("at_risk_pct", ascending=False)
)
print("\nAt-risk % by state (top 10):")
print(at_risk_state.head(10).to_string(index=False))

At-risk breakdown:
  Low rating only (< 3.0):      235
  High late-delivery only (>30%): 141
  Both:                          67

At-risk % by state (top 10):
seller_state  at_risk_count  sellers  at_risk_pct
          AC              1        1   100.000000
          AM              1        1   100.000000
          CE              5       13    38.461538
          PE              2        9    22.222222
          MS              1        5    20.000000
          RN              1        5    20.000000
          BA              3       19    15.789474
          RS             20      129    15.503876
          SP            285     1849    15.413737
          RJ             23      171    13.450292


In [39]:
# Seller segment scatter: revenue vs rating, coloured by segment
active_plot = active.dropna(subset=["avg_review_score"]).copy()
active_plot["segment"] = "Regular"
active_plot.loc[active_plot.power_seller == True, "segment"] = "Power"
active_plot.loc[active_plot.at_risk == True, "segment"] = "At-Risk"

fig = px.scatter(
    active_plot,
    x="total_orders", y="avg_review_score",
    color="segment",
    color_discrete_map={"Power": "#00C851", "At-Risk": "#FF4444", "Regular": "#FF8C00"},
    size="total_revenue", size_max=20,
    opacity=0.6,
    title="Seller Segments: Orders vs Review Score (size = revenue)",
    labels={"total_orders": "Total Orders", "avg_review_score": "Avg Review Score"},
)
fig.add_hline(y=3, line_dash="dash", line_color="#FF4444", annotation_text="At-risk threshold")
fig.update_layout(template="plotly_dark", paper_bgcolor="#0D0D0D", plot_bgcolor="#1A1A1A")
fig.show()